In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as func

import mmap
import random
import pickle

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = 64
batch_size = 128

max_iterations = 3000
learning_rate = 3e-4
eval_interval = 100

n_embed = 384
n_head = 4
n_layer = 4

dropout = 0.1

cuda


In [2]:
vocab = ""
with open("openwebtext/vocab.txt", "r", encoding='utf-8') as f:
    text = f.read()
    vocab = sorted(list(set(text)))

vocab_size = len(vocab)

string_to_int = { ch:i for i, ch in enumerate(vocab) }
int_to_string = { i:ch for i, ch in enumerate(vocab) }

encode = lambda s: [string_to_int[c] for c in s ]
decode = lambda l: ''.join([int_to_string[i] for i in l])

In [3]:
def get_random_chunk(split):
    filename = "openwebtext/output_train.txt" if split == 'train' else "openwebtext/output_valid.txt"

    with open(filename, 'rb') as f:
        with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mem_map:
            file_size = len(mem_map)
            start_pos = random.randint(0, (file_size) - block_size * batch_size)

            mem_map.seek(start_pos)
            
            block = mem_map.read(block_size * batch_size - 1)
            decoded_block = block.decode('utf-8', errors='ignore').replace('\r', '')
            
            data = torch.tensor(encode(decoded_block), dtype=torch.long)

    return data

In [4]:
def get_batch(split):
    data = get_random_chunk(split)
    
    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [5]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()

    for split in ['train', 'validate']:
        losses = torch.zeros(eval_interval)

        for k in range(eval_interval):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()

        out[split] = losses.mean()
    
    model.train()
    return out

In [6]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        BATCH, TIME, CHANNEL = x.shape
        
        key = self.key(x)
        query = self.query(x)

        weights = query @ key.transpose(-2, -1) * key.shape[-1] ** -0.5
        weights = weights.masked_fill(self.tril[:TIME, :TIME] == 0, float('-inf'))
        weights = func.softmax(weights, dim=-1)
        weights = self.dropout(weights)

        value = self.value(x)
        out = weights @ value

        return out

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_head, head_size):
        super().__init__()

        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_head)])
        self.project = nn.Linear(head_size * n_head, n_embed)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.project(out))
        
        return out

In [8]:
class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [9]:
class Block(nn.Module):
    def __init__(self, n_embed, n_head):
        super().__init__()

        head_size = n_embed // n_head

        self.self_attention = MultiHeadAttention(n_head, head_size)
        self.feed_fwd = FeedForward(n_embed)
        self.layer_norm1 = nn.LayerNorm(n_embed)
        self.layer_norm2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        y = self.self_attention(x)
        x = self.layer_norm1(x + y)
        y = self.feed_fwd(x)
        x = self.layer_norm2(x + y)

        return x

In [10]:
class GPTLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        
        self.token_embed_table = nn.Embedding(vocab_size, n_embed)
        self.pos_embed_table   = nn.Embedding(block_size, n_embed)
        
        self.blocks = nn.Sequential(*[Block(n_embed, n_head=n_head) for _ in range(n_layer)])

        self.layer_norm_final = nn.LayerNorm(n_embed)
        self.lang_model_head = nn.Linear(n_embed, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.2)
    
    def forward(self, index, targets=None):
        BATCH, TIME = index.shape
        
        token_embed = self.token_embed_table(index)
        pos_embed = self.pos_embed_table(torch.arange(TIME, device=device))

        x = token_embed + pos_embed
        x = self.blocks(x)
        x = self.layer_norm_final(x)

        logits = self.lang_model_head(x)
        
        if targets == None:
            loss = None
        else:
            BATCH, TIME, CHANNEL = logits.shape
            logits = logits.view(BATCH * TIME, CHANNEL)
            targets = targets.view(BATCH * TIME)
            loss = func.cross_entropy(logits, targets)
            
        return logits, loss

    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self.forward(index)
            
            logits = logits[:, -1, :]
            probs = func.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)

        return index

model = GPTLM(vocab_size)

print('loading model params...')
with open('model-01.pkl', 'rb') as f:
    model = pickle.load()
print('loaded successfully!')

m = model.to(device)

In [11]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iteration in range(max_iterations):
    if iteration % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iteration:4d} | train: {losses['train']:.4f} | val: {losses['validate']:.4f}")
    
    xb, yb = get_batch('train')
    logits, loss = model.forward(xb, yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

with open('model-01.pkl', 'wb') as f:
    pickle.dump(model, f)

print('model saved')

step    0 | train: 10.4235 | val: 10.4237
step  100 | train: 2.7260 | val: 2.7576
step  200 | train: 2.5867 | val: 2.6100
step  300 | train: 2.3695 | val: 2.3879
step  400 | train: 2.2429 | val: 2.2653
step  500 | train: 2.2273 | val: 2.2136
step  600 | train: 2.1347 | val: 2.1348
step  700 | train: 2.0321 | val: 2.0536
step  800 | train: 1.9944 | val: 2.0259
step  900 | train: 2.0015 | val: 1.9452
step 1000 | train: 1.9542 | val: 1.9990
step 1100 | train: 2.0092 | val: 1.9291
step 1200 | train: 1.8697 | val: 1.8897
step 1300 | train: 1.8929 | val: 1.8437
step 1400 | train: 1.8259 | val: 1.8434
step 1500 | train: 1.8072 | val: 1.7863
step 1600 | train: 1.8100 | val: 1.7956
step 1700 | train: 1.7881 | val: 1.8478
step 1800 | train: 1.8035 | val: 1.7818
step 1900 | train: 1.7347 | val: 1.7757
step 2000 | train: 1.7781 | val: 1.7477
step 2100 | train: 1.7538 | val: 1.7122
step 2200 | train: 1.7404 | val: 1.7561
step 2300 | train: 1.7319 | val: 1.7230
step 2400 | train: 1.7013 | val: 1.700